In [35]:
from pathlib import Path
import geopandas as gpd

# 1. Setup paths
geodata_path = Path.cwd().parent / "data/geo/raw"
shapefiles = list(geodata_path.glob("**/*.shp"))

# 2. Settings
TARGET_EPSG = 32615  # UTM Zone 15N
final_gdfs = {}

print(f"{'Filename':<30} | {'Rows':<6} | {'CRS Status'}")
print("-" * 60)

# 3. Single-pass processing
for file_path in shapefiles:
    name = file_path.stem
    
    # Load and immediately re-project to avoid redundant memory usage
    gdf = gpd.read_file(file_path).to_crs(epsg=TARGET_EPSG)
    
    # Add Area Columns (Calculated once since both use same base math)
    sq_meters = gdf.geometry.area
    gdf['area_m2'] = sq_meters
    gdf['area_km2'] = sq_meters / 1_000_000
    
    # Store in dictionary
    final_gdfs[name] = gdf
    
    # Inline Progress Check
    print(f"{name:<30} | {len(gdf):<6} | Unified to {TARGET_EPSG}")

# 4. Verify specific layer
if 'mo_vest_20' in final_gdfs:
    print("\nVerification for mo_vest_20:")
    print(final_gdfs['mo_vest_20'][['area_m2', 'area_km2']].head())

Filename                       | Rows   | CRS Status
------------------------------------------------------------
mo_2010_cnty_bound             | 115    | Unified to 32615
mo_vest_16                     | 3324   | Unified to 32615
mo_cnty_2020_bound             | 115    | Unified to 32615
mo_vest_20                     | 3733   | Unified to 32615
mo_2024_gen_all_prec           | 3333   | Unified to 32615
mo_2024_gen_cong_prec          | 3357   | Unified to 32615
mo_2024_gen_sldl_prec          | 3573   | Unified to 32615
mo_2024_gen_sldu_prec          | 3335   | Unified to 32615

Verification for mo_vest_20:
        area_m2   area_km2
0  9.149077e+07  91.490775
1  3.318367e+07  33.183667
2  6.148763e+06   6.148763
3  9.835783e+06   9.835783
4  2.108673e+06   2.108673
